In [1]:
from case_studies.lotka_volterra_UDE_case_study.lotka_volterra_UDE_case_study.mod import Func
import jax.random as jrandom
import jax.nn as jnn
import jax.numpy as jnp
from case_studies.lotka_volterra_UDE_case_study.lotka_volterra_UDE_case_study.sim import UDESimulation
from pymob import Config
import jax.tree_util as jtu

In [2]:
t = jnp.arange(20)
y = jnp.array([40,9])
alpha = 0.5
beta = 0.03
gamma = 0.02
delta = 0.5

key = jrandom.PRNGKey(5678)
data_key, model_key, loader_key = jrandom.split(key, 3)
func = Func(10,10,key=model_key,theta_true=jnp.array([alpha,beta,gamma]))

In [3]:
func(t,y)

Array([9.200001 , 6.8116455], dtype=float32)

In [3]:
config = Config("case_studies/lotka_volterra_UDE_case_study/scenarios/test_scenario_v2/settings.cfg")
config.case_study.package = ".."

sim = UDESimulation(config)
sim.setup()

c:\Users\Markus\pymob\pymob\pymob\simulation.py:444: UserWarning: Module sim.py not found in case_studies or in case_studies. Missing modules can lead to unexpected behavior. Does your case study of the parent class have a sim.py file? It should have the line `from PARENT_CASE_STUDY. sim import *` to import all objects from the parent case study.
  warnings.warn(
c:\Users\Markus\pymob\pymob\pymob\simulation.py:444: UserWarning: Module mod.py not found in case_studies or in case_studies. Missing modules can lead to unexpected behavior. Does your case study of the parent class have a mod.py file? It should have the line `from PARENT_CASE_STUDY. mod import *` to import all objects from the parent case study.
  warnings.warn(
c:\Users\Markus\pymob\pymob\pymob\simulation.py:444: UserWarning: Module prob.py not found in case_studies or in case_studies. Missing modules can lead to unexpected behavior. Does your case study of the parent class have a prob.py file? It should have the line `from 

Results directory exists at 'c:\Users\Markus\pymob\lotka_volterra_UDE_case_study\results\test_scenario_v2'.
Scenario directory exists at 'c:\Users\Markus\pymob\lotka_volterra_UDE_case_study\scenarios\test_scenario_v2'.
MinMaxScaler(variable=rabbits, min=5.968110437683305, max=86.99133665713266)
MinMaxScaler(variable=wolves, min=7.203778019337644, max=62.829641338400535)


TypeError: unhashable type: 'jaxlib._jax.ArrayImpl'

In [10]:
type(jtu.tree_leaves(sim.model)[0])

jaxlib.xla_extension.ArrayImpl

In [ ]:
import equinox as eqx
import jax.random as jr
import jax

In [12]:
mlp: eqx.nn.MLP
key = jr.PRNGKey(5678)
data_key, model_key, loader_key = jr.split(key, 3)
mlp = eqx.nn.MLP(in_size = 2, out_size = 2, width_size = 64, depth = 3, key = model_key)

#mlp(jnp.array([0,0]))

In [30]:
get_bias(mlp)[0].shape

(64,)

In [66]:
in_size = 2
out_size = 2
width_size = 3
depth = 3

In [93]:
key = jr.PRNGKey(5678)
data_key, model_key, loader_key = jr.split(key, 3)
mlp = eqx.nn.MLP(in_size = in_size, out_size = out_size, width_size = width_size, depth = depth, key = model_key)

mlp(jnp.array([1,1]))

Array([ 0.00508419, -0.52783597], dtype=float32)

In [108]:
def transformWeights(weights):
    depth = len(weights)-1
    width_size, in_size = weights[0].shape
    out_size = weights[-1].shape[0]
    list = []
    for layer in weights:
        dims = layer.shape
        elms = dims[0] * dims[1]
        layerR = layer.reshape(elms)
        for el in layerR:
            list.append(el.item())
    return in_size, out_size, width_size, depth, list

In [105]:
def transformWeightsBackwards(in_size, out_size, width_size, depth, list):
    countLayer = 0
    countElms = 0
    res = []
    while (countLayer <= depth):
        if countLayer == 0:
            elms = in_size * width_size
            weights = jnp.array(list[countElms:countElms+elms]).reshape((width_size,in_size))
            countElms += elms
            countLayer += 1
            res.append(weights)
        elif countLayer == depth:
            elms = width_size * out_size
            weights = jnp.array(list[countElms:countElms+elms]).reshape((out_size,width_size))
            countElms += elms
            countLayer += 1
            res.append(weights)
        else:
            elms = width_size * width_size
            weights = jnp.array(list[countElms:countElms+elms]).reshape((width_size,width_size))
            countElms += elms
            countLayer += 1
            res.append(weights)
    return res

In [106]:
def transformBias(bias):
    depth = len(bias)-1
    width_size = len(bias[0])
    out_size = len(bias[-1])
    list = []
    for layer in bias:
        for el in layer:
            list.append(el.item())
    return out_size, width_size, depth, list

In [107]:
def transformBiasBackwards(out_size, width_size, depth, list):
    countLayer = 0
    countElms = 0
    res = []
    while (countLayer <= depth):
        if countLayer == depth:
            elms = out_size
            bias = jnp.array(list[countElms:countElms+elms])
            countElms += elms
            countLayer += 1
            res.append(bias)
        else:
            elms = width_size
            bias = jnp.array(list[countElms:countElms+elms])
            countElms += elms
            countLayer += 1
            res.append(bias)
    return res

In [11]:
is_linear = lambda x: isinstance(x, eqx.nn.Linear)
get_weights = lambda m: [x.weight
                        for x in jax.tree_util.tree_leaves(m, is_leaf=is_linear)
                        if is_linear(x)]
weights = get_weights(mlp)

In [26]:
is_linear = lambda x: isinstance(x, eqx.nn.Linear)
get_bias = lambda m: [x.bias
                        for x in jax.tree_util.tree_leaves(m, is_leaf=is_linear)
                        if is_linear(x)]
bias = get_bias(mlp)

In [104]:
mlp.width_size

3

In [109]:
def init_custom_weight(model, list):
  is_linear = lambda x: isinstance(x, eqx.nn.Linear)
  get_weights = lambda m: [x.weight
                           for x in jax.tree_util.tree_leaves(m, is_leaf=is_linear)
                           if is_linear(x)]
  new_weights = transformWeightsBackwards(in_size = mlp.in_size, out_size = mlp.out_size, width_size = mlp.width_size, depth = mlp.depth, list = list)
  new_model = eqx.tree_at(get_weights, model, new_weights)
  return new_model

In [110]:
new_weights = [0.5] * 30

new_mlp = init_custom_weight(mlp, new_weights)

In [111]:
get_weights(new_mlp)

[Array([[0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5]], dtype=float32),
 Array([[0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]], dtype=float32),
 Array([[0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]], dtype=float32),
 Array([[0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]], dtype=float32)]

In [24]:
def test (a, b, *args):
    return list(args[1:])

In [25]:
test(1,2,3,4,5)

[4, 5]

In [29]:
a=1
b=1

list((a,b))

[1, 1]